# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain in simple language what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get gpt-4o-mini to answer, with streaming

def stream_technical_answer(question):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers technical questions"},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_technical_answer(question)

What the code does

- It’s inside a generator function (because it uses yield from).
- The inner part {book.get("author") for book in books if book.get("author")} is a set comprehension:
  - It iterates over books (each book is assumed to be a dict-like object).
  - For each book, it gets the value of the "author" key (book.get("author")).
  - The if book.get("author") filters out any falsy author values (e.g., None or empty string).
  - The result is a set of unique author values (duplicates are removed because it's a set).
- yield from <set> yields each element of that set to the caller of the generator. So the generator will yield each unique, non-empty author exactly once.

Why you’d write it this way

- To produce a sequence of unique authors from a collection of books, skipping entries that don’t have an author.
- Using a set ensures deduplication in a concise way, and yield from streams those authors to the caller without building an extra list.

Notes and caveats

- The order of yielded authors is not deterministic because sets are unordered. If you need a deterministic order (e.g., order of first appearance), use a different approach (e.g., track seen authors with a set and yield in a loop, preserving order).
- If books isn’t a sequence of dict-like objects, or authors aren’t hashable, this could raise errors.
- Example behavior:
  - Given books = [{ "title": "X", "author": "A" }, { "title": "Y", "author": "B" }, { "title": "Z", "author": "A" }, { "title": "W" }], the generator will yield "A" and "B" (order unspecified).

In [ ]:
# Get Llama 3.2 to answer